# Grid Placement -> Final Positions

### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.ndimage import gaussian_filter

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

### Loading dataset

In [2]:
df = pd.read_csv("../../data/positional_data.csv")
df.head()

,raceId,driverId,constructorId,grid,position,positionText,rank,points,time,statusId,code,surname,forename,year,circuitId,date,name,round,status
0,1,18,23,1,1,1,3,10.0,1:34:15.784,1,BUT,Button,Jenson,2009,1,2009-03-29,Australian Grand Prix,1,Finished
1,1,22,23,2,2,2,14,8.0,+0.807,1,BAR,Barrichello,Rubens,2009,1,2009-03-29,Australian Grand Prix,1,Finished
2,1,20,9,3,13,13,4,0.0,NaN,4,VET,Vettel,Sebastian,2009,1,2009-03-29,Australian Grand Prix,1,Collision
3,1,9,2,4,14,14,2,0.0,NaN,4,KUB,Kubica,Robert,2009,1,2009-03-29,Australian Grand Prix,1,Collision
4,1,3,3,5,6,6,1,3.0,+5.722,1,ROS,Rosberg,Nico,2009,1,2009-03-29,Australian Grand Prix,1,Finished


### Replacing non-standard positions

In [3]:
df.loc[df['grid'] > 20, 'grid'] = '20+'
df.loc[df['grid'] == 0, 'grid'] = 'Pit'

mask = (df['position'] > 20)
df.loc[mask, 'position'] = df.loc[mask, 'position'].apply(lambda x: 'DNF' if x == 99 else '20+')

### String Labels for plot

In [4]:
grid_labels = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '20+', 'Pit']
position_labels = ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '20+', 'DNF']

### Typecasting to strings for consistency

In [5]:
df['grid'] = df['grid'].astype(str)
df['position'] = df['position'].astype(str)

### Driver List

Retaining drivers with 50 or more race appearances

In [6]:
driver_info = df[['driverId', 'code', 'forename', 'surname']].copy()

# Drop drivers with code '\N'
driver_info = driver_info[driver_info['code'] != '\\N']

# Dropping drivers with less than 50 appearances
driver_counts = driver_info['driverId'].value_counts()
filtered_drivers = driver_counts[driver_counts >= 50].index.tolist()
driver_info = driver_info[driver_info['driverId'].isin(filtered_drivers)]

# Sort by driverId
driver_info = driver_info.sort_values('driverId')

# Get full name
driver_info['full_name'] = driver_info['forename'] + ' ' + driver_info['surname']

# Store unique values in dictionary
driver_info = driver_info.drop_duplicates(subset=['driverId']).sort_values('full_name')
driver_dict = dict(zip(driver_info['driverId'], driver_info['full_name']))

driver_dict

{16: 'Adrian Sutil',
 848: 'Alexander Albon',
 25: 'Alexander Wurz',
 841: 'Antonio Giovinazzi',
 832: 'Carlos Sainz',
 844: 'Charles Leclerc',
 32: 'Christian Klien',
 817: 'Daniel Ricciardo',
 826: 'Daniil Kvyat',
 14: 'David Coulthard',
 821: 'Esteban Gutiérrez',
 839: 'Esteban Ocon',
 13: 'Felipe Massa',
 4: 'Fernando Alonso',
 847: 'George Russell',
 21: 'Giancarlo Fisichella',
 855: 'Guanyu Zhou',
 5: 'Heikki Kovalainen',
 35: 'Jacques Villeneuve',
 15: 'Jarno Trulli',
 818: 'Jean-Éric Vergne',
 18: 'Jenson Button',
 31: 'Juan Pablo Montoya',
 155: 'Kamui Kobayashi',
 825: 'Kevin Magnussen',
 8: 'Kimi Räikkönen',
 840: 'Lance Stroll',
 846: 'Lando Norris',
 1: 'Lewis Hamilton',
 69: 'Luca Badoer',
 828: 'Marcus Ericsson',
 17: 'Mark Webber',
 830: 'Max Verstappen',
 30: 'Michael Schumacher',
 849: 'Nicholas Latifi',
 2: 'Nick Heidfeld',
 807: 'Nico Hülkenberg',
 3: 'Nico Rosberg',
 813: 'Pastor Maldonado',
 814: 'Paul di Resta',
 37: 'Pedro de la Rosa',
 842: 'Pierre Gasly',
 23:

### Filtering dataframe based on driver list

In [7]:
df = df[df['driverId'].isin(driver_dict.keys())]

print(df['grid'].value_counts().sort_values(ascending=False))
print(df['position'].value_counts().sort_values(ascending=False))

grid
3      533
1      521
2      515
4      504
5      502
6      502
7      489
9      481
8      469
10     455
11     451
12     434
13     423
14     422
16     386
15     363
17     344
18     324
19     298
20     274
20+    267
Pit     81
Name: count, dtype: int64
position
DNF    1885
1       523
2       520
3       504
5       498
4       498
6       470
7       464
8       440
9       439
10      423
11      398
12      368
13      343
14      315
15      272
16      228
17      187
18      129
19       75
20       31
20+      28
Name: count, dtype: int64


### Overall transition matrix for all drivers

In [8]:
g_transition_matrix = pd.crosstab(df['grid'], df['position'], normalize='all') * 100

# Reorder the matrix based on grid_labels and position_labels
g_transition_matrix = g_transition_matrix.reindex(index=grid_labels, columns=position_labels).round(3)

g_transition_matrix

position,1,2,3,4,5,6,7,8,9,10,...,13,14,15,16,17,18,19,20,20+,DNF
grid,,,,,,,,,,,,,,,,,,,,,
1,2.844,1.062,0.520,0.221,0.122,0.055,0.033,0.055,0.055,0.011,...,0.011,0.022,0.011,0.000,0.000,0.000,0.000,0.000,0.000,0.719
2,1.461,1.438,0.741,0.387,0.277,0.100,0.188,0.077,0.089,0.011,...,0.011,0.011,0.011,0.011,0.011,0.000,0.033,0.011,0.000,0.741
3,0.686,1.184,1.228,0.586,0.465,0.254,0.155,0.111,0.077,0.089,...,0.033,0.044,0.011,0.000,0.033,0.011,0.000,0.000,0.000,0.863
4,0.254,0.786,0.830,0.985,0.564,0.432,0.299,0.144,0.100,0.055,...,0.033,0.033,0.044,0.011,0.022,0.033,0.022,0.000,0.000,0.863
5,0.111,0.387,0.597,0.852,0.719,0.487,0.299,0.221,0.133,0.166,...,0.000,0.100,0.033,0.044,0.033,0.033,0.000,0.000,0.011,1.162
6,0.144,0.254,0.531,0.609,0.664,0.752,0.432,0.254,0.266,0.155,...,0.044,0.044,0.089,0.033,0.033,0.011,0.000,0.000,0.011,1.040
7,0.077,0.177,0.266,0.288,0.620,0.697,0.609,0.420,0.288,0.243,...,0.122,0.044,0.100,0.066,0.033,0.011,0.022,0.000,0.000,0.985
8,0.033,0.089,0.199,0.398,0.443,0.376,0.542,0.432,0.420,0.288,...,0.199,0.044,0.100,0.022,0.055,0.011,0.000,0.011,0.000,1.095
9,0.011,0.089,0.100,0.321,0.398,0.487,0.553,0.509,0.487,0.332,...,0.199,0.100,0.133,0.033,0.000,0.000,0.011,0.000,0.000,1.140


### Transition Matrix by driver

In [9]:
driver_matrix = {}
driver_count_matrix = {}
for driverId in driver_dict.keys():
    driver_df = df[df['driverId'] == driverId].copy()
    driver_matrix[driverId] = (pd.crosstab(driver_df['grid'], driver_df['position'], normalize='all').reindex(index=grid_labels, columns=position_labels, fill_value=0) * 100).round(3)
    driver_count_matrix[driverId] = pd.crosstab(driver_df['grid'], driver_df['position']).reindex(index=grid_labels, columns=position_labels, fill_value=0)


### Surface Plot

#### Unsmoothed

In [10]:
fig = go.Figure()

# matrix layout
X, Y = np.meshgrid(position_labels, grid_labels)

# Traces for each driver
surface_colormap = {id: np.zeros_like(g_transition_matrix.loc[grid_labels, position_labels]) for id in driver_dict.keys() }

for driverId, driver_data in driver_matrix.items():
    for i, grid in enumerate(grid_labels):
        for j, position in enumerate(position_labels):
            if i >= j:  # Lower triangle
                surface_colormap[driverId][i, j] = (driver_data.loc[grid, position] - g_transition_matrix.loc[grid, position]) if driver_data.loc[grid, position] else 0
            else:       # Upper triangle
                surface_colormap[driverId][i, j] = -(driver_data.loc[grid, position] - g_transition_matrix.loc[grid, position]) if driver_data.loc[grid, position] else 0


g_cmin = min([float(np.min(matrix)) for id, matrix in surface_colormap.items()])
g_cmax = max([float(np.max(matrix)) for id, matrix in surface_colormap.items()])


for driverId, driver_data in driver_matrix.items():
    fig.add_trace(go.Surface(
          z=driver_data.loc[grid_labels, position_labels].values,
          x=X,
          y=Y,
          surfacecolor=surface_colormap[driverId],
          colorscale=[[0, '#fcb60c'], [0.5, '#808080'], [1, '#ff1e00']],
          colorbar=dict(title='Relative performance difference in %  vs.  all drivers', orientation='v', titleside='right'),
          # cmin=g_cmin,
          cmid=0,
          # cmax=g_cmax,
          opacity=0.95,
          showscale=True,
          name=driver_dict[driverId],
          customdata=np.transpose(driver_count_matrix[driverId].values),
          hovertemplate="%{customdata} results (%{z}%) at <br>Grid Spot  \t\t\t\t\t\t: %{y}<br>Final Position : %{x}<br><extra></extra>",
          visible=(driverId == 1)  # Default: HAM visible
      ))


# Dropdown menu setup
driver_select = []
active_index = 0
for idx, (driverId, driver_name) in enumerate(driver_dict.items()):
    visibility = [(j == idx) for j in range(len(driver_dict))]

    button = dict(
        label=driver_name,
        method="update",
        args=[
            { "visible": visibility },
            { "title": {
                  "text": f"Performance across various positions - {driver_name}",
                  "font": {'size': 20, 'family': 'Arial'},
                  "x": 0.5,
                  "y": 0.98,
                  "xanchor": "center"
              }
            }
        ]
    )
    driver_select.append(button)
    if driverId == 1:
        active_index = idx

# Update layout
fig.update_layout(
    updatemenus=[
        dict(
            buttons=driver_select,
            direction="down",
            showactive=True,
            active=active_index,
            x=0.14,
            xanchor="left",
            y=1.05,
            yanchor="top",
            bgcolor='rgba(50, 50, 50, 0.8)',
            bordercolor='lightgray',
            font=dict(color='#ff1e00')
        )
    ],
    annotations=[
        dict(text="Select Driver :", x=0, xref="paper", y=1.043, yref="paper", align="left", showarrow=False)
    ],
    title={
        "text": "Performance across various positions - Lewis Hamilton",  # Default title
        "font": {'size': 20, 'family': 'Arial'},
        "x": 0.5,
        "y": 0.98,
        "xanchor": "center",
    },
    scene=dict(
        xaxis_title='Final Position',
        yaxis_title='Grid Spot',
        zaxis_title='Percentage of results (%)',
        xaxis=dict(showbackground=True),  # Show/Hide XZ plane
        yaxis=dict(showbackground=True),  # Show/Hide YZ plane
        zaxis=dict(range=[0, 20]),
        camera=dict(
            eye=dict(x=1.3, y=1.3, z=1.5)  # Adjust these values
        )
    ),
    width=800,
    height=800,
    template="plotly_dark",
    margin=dict(l=10, r=0, b=10, t=80)
)

fig.show()

### Heatmap (not used)

In [60]:
import altair as alt

def plot_transition_heatmap(df, driver='HAM'):
    if driver:
        driver_df = df[df['code'] == driver].copy()

    # create transition matrix
    transition_matrix = pd.crosstab(df['grid'], df['position'], normalize='index')
    transition_matrix = transition_matrix.reindex(index=grid_labels, columns=position_labels)

    # melt into long format for Altair
    plot_data = transition_matrix.reset_index().melt(id_vars='grid', var_name='position', value_name='probability')

    # plot
    heatmap = alt.Chart(plot_data).mark_rect().encode(
        x=alt.X('position:O', title='Final Position'),
        y=alt.Y('grid:O', title='Starting Grid Position', sort='ascending'),  # ascending so top is P1
        color=alt.Color('probability:Q', scale=alt.Scale(scheme='blues')),
        tooltip=['grid', 'position', alt.Tooltip('probability:Q', format='.2f')]
    ).properties(
        width=400,
        height=400,
        title=f'Transition Matrix: {driver}'
    )

    return heatmap